# 02 — Build the particle-level PyTorch dataset
This notebook turns the validated EDM4HEP `Particle` collection into a dataset for masked-particle pretraining.

**Baseline design:** generator-level final-state particles (`generatorStatus == 1`), nonzero 3-momentum, deterministic pT ordering, 7 continuous features, and a categorical PDG token.

In [1]:
import numpy as np
import awkward as ak
import uproot
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.11.0
CUDA available: True
GPU: Tesla T4


In [2]:
FILE = "clean_data_ecm240.root"
events = uproot.open(FILE)["events"]
print(events)
print("Events:", events.num_entries)


<TTree 'events' (10 branches) at 0x7f6f2a02fb60>
Events: 45830


In [3]:
branches = [
    "Particle.PDG",
    "Particle.generatorStatus",
    "Particle.charge",
    "Particle.mass",
    "Particle.momentum.x",
    "Particle.momentum.y",
    "Particle.momentum.z",
]

particle = events.arrays(branches, library="ak")
print(particle.fields)


['Particle.PDG', 'Particle.generatorStatus', 'Particle.charge', 'Particle.mass', 'Particle.momentum.x', 'Particle.momentum.y', 'Particle.momentum.z']


In [14]:
# 1. Keep generator-level final-state particles.
final_mask = particle["Particle.generatorStatus"] == 1

pdg = particle["Particle.PDG"][final_mask]
charge = particle["Particle.charge"][final_mask]
mass = particle["Particle.mass"][final_mask]
px = particle["Particle.momentum.x"][final_mask]
py = particle["Particle.momentum.y"][final_mask]
pz = particle["Particle.momentum.z"][final_mask]

# 2. Derived kinematics.
pt = np.sqrt(px**2 + py**2)
phi = np.arctan2(py, px)

energy = np.sqrt(px**2 + py**2 + pz**2 + mass**2)

# 3. Keep only particles with nonzero transverse momentum.
#
# Our current particle representation uses eta and phi,
# so pT = 0 cannot be represented cleanly in these coordinates.

nonzero_pt = pt > 0

pdg = pdg[nonzero_pt]
charge = charge[nonzero_pt]
mass = mass[nonzero_pt]

px = px[nonzero_pt]
py = py[nonzero_pt]
pz = pz[nonzero_pt]

pt = pt[nonzero_pt]
phi = phi[nonzero_pt]
energy = energy[nonzero_pt]

# 4. Pseudorapidity is now finite by construction.
eta = np.arcsinh(pz / pt)

multiplicity = ak.to_numpy(ak.num(pdg, axis=1))

print("After pT > 0 selection:")
print("  min   :", multiplicity.min())
print("  max   :", multiplicity.max())
print("  mean  :", multiplicity.mean())
print("  median:", np.median(multiplicity))

After pT > 0 selection:
  min   : 33
  max   : 173
  mean  : 80.35852061968143
  median: 79.0


In [15]:
# Sort particles by descending pT.
# Collider events are naturally sets; this gives our first prototype a deterministic sequence.
order = ak.argsort(pt, axis=1, ascending=False)

pdg = pdg[order]
charge = charge[order]
mass = mass[order]
px = px[order]
py = py[order]
pz = pz[order]
pt = pt[order]
phi = phi[order]
energy = energy[order]
eta = eta[order]

print("First event pT:", pt[0])
print("First event PDG:", pdg[0])


First event pT: [37.4, 8.12, 6.9, 5.82, 5.5, 5, ..., 0.05, 0.0469, 0.0458, 0.0335, 0.0205]
First event PDG: [310, -211, 130, 22, -321, 22, -211, 22, ..., 22, 22, 22, 22, -211, 22, 22, 22]


## Particle representation
We use the continuous vector

$$x_i=[\log(1+p_T),\eta,\sin\phi,\cos\phi,\log(1+E),\log(1+m),q].$$

PDG ID stays categorical and will later be represented by a learned embedding. Using sine/cosine avoids the artificial `-π/π` discontinuity.

In [16]:
continuous = ak.zip({
    "log_pt": np.log1p(pt),
    "eta": eta,
    "sin_phi": np.sin(phi),
    "cos_phi": np.cos(phi),
    "log_energy": np.log1p(energy),
    "log_mass": np.log1p(mass),
    "charge": charge,
})

print("First five particles of event 0:")
print(continuous[0][:5])


First five particles of event 0:
[{log_pt: 3.65, eta: 1.02, sin_phi: -0.621, cos_phi: -0.784, ...}, ..., {...}]


In [17]:
# Build a compact categorical PDG vocabulary.
pdg_flat = ak.to_numpy(ak.flatten(pdg, axis=1))
unique_pdg = np.unique(pdg_flat)
pdg_to_id = {int(p): i + 1 for i, p in enumerate(unique_pdg)}
PAD_ID = 0
UNK_ID = 0

pdg_id = ak.Array([
    [pdg_to_id[int(x)] for x in event]
    for event in ak.to_list(pdg)
])

print("Unique PDGs:", len(unique_pdg))
print("PDG vocabulary size including PAD/UNK:", len(pdg_to_id) + 1)
print("First event PDG tokens:", pdg_id[0])


Unique PDGs: 31
PDG vocabulary size including PAD/UNK: 32
First event PDG tokens: [23, 9, 21, 20, 8, 20, 9, 20, 9, 20, ..., 20, 22, 20, 20, 20, 20, 9, 20, 20, 20]


In [18]:
# Keep variable-length events until batching. Padding belongs in the collate function.
continuous_list = ak.to_list(continuous)
pdg_id_list = ak.to_list(pdg_id)

print("Events:", len(continuous_list))
print("First event length:", len(continuous_list[0]))
print("First particle:", continuous_list[0][0])
print("First particle PDG token:", pdg_id_list[0][0])


Events: 45830
First event length: 68
First particle: {'log_pt': 3.6491713523864746, 'eta': 1.0238126516342163, 'sin_phi': -0.6211046576499939, 'cos_phi': -0.7837276458740234, 'log_energy': 4.091735654989596, 'log_mass': 0.4038705059096693, 'charge': 0.0}
First particle PDG token: 23


In [19]:
# Event-level split: no particles from one event can appear in multiple splits.
rng = np.random.default_rng(42)
n_events = len(continuous_list)
indices = np.arange(n_events)
rng.shuffle(indices)

n_train = int(0.80 * n_events)
n_val = int(0.10 * n_events)
train_idx = indices[:n_train]
val_idx = indices[n_train:n_train + n_val]
test_idx = indices[n_train + n_val:]

print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))


Train: 36664
Validation: 4583
Test: 4583


In [20]:
# Compute normalization from TRAINING particles only.
train_rows = []
for i in train_idx:
    for p in continuous_list[int(i)]:
        train_rows.append([
            p["log_pt"], p["eta"], p["sin_phi"], p["cos_phi"],
            p["log_energy"], p["log_mass"], p["charge"]
        ])

train_features = np.asarray(train_rows, dtype=np.float32)
feature_mean = np.nanmean(train_features, axis=0)
feature_std = np.nanstd(train_features, axis=0)
feature_std = np.where(feature_std < 1e-6, 1.0, feature_std)

print("Feature means:", feature_mean)
print("Feature stds :", feature_std)


Feature means: [7.88124204e-01 1.17501244e-03 5.82589942e-04 4.07793472e-04
 9.79417384e-01 1.13426484e-01 0.00000000e+00]
Feature stds : [0.7050907  1.1159818  0.7078694  0.70634246 0.7873272  0.16751803
 0.6844187 ]


In [21]:
class ColliderEventDataset(torch.utils.data.Dataset):
    def __init__(self, event_indices, feature_lists, pdg_lists, mean, std):
        self.event_indices = np.asarray(event_indices)
        self.feature_lists = feature_lists
        self.pdg_lists = pdg_lists
        self.mean = np.asarray(mean, dtype=np.float32)
        self.std = np.asarray(std, dtype=np.float32)

    def __len__(self):
        return len(self.event_indices)

    def __getitem__(self, idx):
        event_index = int(self.event_indices[idx])
        rows = []
        for p in self.feature_lists[event_index]:
            rows.append([
                p["log_pt"], p["eta"], p["sin_phi"], p["cos_phi"],
                p["log_energy"], p["log_mass"], p["charge"]
            ])
        features = np.asarray(rows, dtype=np.float32)
        features = (features - self.mean) / self.std
        pdg_tokens = np.asarray(self.pdg_lists[event_index], dtype=np.int64)

        return {
            "features": torch.from_numpy(features),
            "pdg": torch.from_numpy(pdg_tokens),
        }

train_dataset = ColliderEventDataset(train_idx, continuous_list, pdg_id_list, feature_mean, feature_std)
val_dataset = ColliderEventDataset(val_idx, continuous_list, pdg_id_list, feature_mean, feature_std)
test_dataset = ColliderEventDataset(test_idx, continuous_list, pdg_id_list, feature_mean, feature_std)

sample = train_dataset[0]
print("Sample features:", sample["features"].shape)
print("Sample PDG tokens:", sample["pdg"].shape)


Sample features: torch.Size([59, 7])
Sample PDG tokens: torch.Size([59])


In [22]:
def collate_events(batch):
    lengths = torch.tensor([item["features"].shape[0] for item in batch], dtype=torch.long)

    features = torch.nn.utils.rnn.pad_sequence(
        [item["features"] for item in batch], batch_first=True, padding_value=0.0
    )
    pdg = torch.nn.utils.rnn.pad_sequence(
        [item["pdg"] for item in batch], batch_first=True, padding_value=PAD_ID
    )

    padding_mask = torch.arange(features.shape[1])[None, :] >= lengths[:, None]

    return {
        "features": features,
        "pdg": pdg,
        "padding_mask": padding_mask,
        "lengths": lengths,
    }

loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=64, shuffle=True, collate_fn=collate_events
)
batch = next(iter(loader))

print("features:", batch["features"].shape)
print("pdg:", batch["pdg"].shape)
print("padding mask:", batch["padding_mask"].shape)
print("lengths:", batch["lengths"][:10])


features: torch.Size([64, 118, 7])
pdg: torch.Size([64, 118])
padding mask: torch.Size([64, 118])
lengths: tensor([ 91,  79,  72,  94,  68,  74, 102,  89,  89,  53])


In [23]:
# Sanity checks.
assert batch["features"].ndim == 3
assert batch["features"].shape[2] == 7
assert batch["pdg"].shape[:2] == batch["features"].shape[:2]
assert batch["padding_mask"].shape == batch["pdg"].shape
assert torch.isfinite(batch["features"]).all()

print("All dataset sanity checks passed.")
print("Maximum sequence length in this batch:", batch["features"].shape[1])


All dataset sanity checks passed.
Maximum sequence length in this batch: 118


In [24]:
# Save the processed dataset and metadata for Notebook 03.

processed_data = {
    "continuous_list": continuous_list,
    "pdg_id_list": pdg_id_list,

    "train_idx": train_idx,
    "val_idx": val_idx,
    "test_idx": test_idx,

    "feature_mean": feature_mean,
    "feature_std": feature_std,

    "pdg_to_id": pdg_to_id,

    "PAD_ID": PAD_ID,
    "VOCAB_SIZE": len(pdg_to_id) + 1,
}

OUTPUT_FILE = "processed_collider_events.pt"

torch.save(processed_data, OUTPUT_FILE)

print(f"Saved processed dataset to: {OUTPUT_FILE}")

Saved processed dataset to: processed_collider_events.pt


## Next: Notebook 03
Notebook 03 will add a 40% masked-particle objective and a small Transformer encoder. We will first train on this dataset without adding unnecessary complexity.